In [2]:
import os
import numpy as np
import pandas as pd
import cv2

from math import ceil
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2


# Path Set
current_dir = os.getcwd()
if os.path.basename(current_dir).lower() == "notebooks":
    BASE_DIR = os.path.dirname(current_dir)
else:
    BASE_DIR = current_dir 

DATASET_DIR = os.path.join(BASE_DIR, "dataset")
IMAGES_DIR = os.path.join(DATASET_DIR, "images")
LABELS_PATH = os.path.join(DATASET_DIR, "labels.csv")
MODELS_DIR = os.path.join(BASE_DIR, "models")

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("CURRENT_DIR:", current_dir)
print("BASE_DIR   :", BASE_DIR)
print("DATASET_DIR:", DATASET_DIR)
print("IMAGES_DIR :", IMAGES_DIR)
print("LABELS_PATH:", LABELS_PATH)
print("MODELS_DIR :", MODELS_DIR)

# ---- 5. Load labels.csv (kalau belum ada, kasih instruksi lalu stop) ----
if not os.path.exists(LABELS_PATH):
    print("\n[INFO] labels.csv not found.")
    print("Buat file di:", LABELS_PATH)
    print("Dengan minimal kolom: filename, muac_cm")
    print("Contoh baris:")
    print("filename,muac_cm")
    print("img_0001.jpg,11.5")
    raise SystemExit("Stop dulu: siapkan labels.csv dan gambar di folder images/")

df = pd.read_csv(LABELS_PATH)

if df.empty:
    print("\n[INFO] labels.csv ada, tapi belum ada baris data.")
    print("Isi dulu data filename + muac_cm sebelum training.")
    raise SystemExit("Stop dulu: belum ada data MUAC untuk training.")

required_cols = {"filename", "muac_cm"}
if not required_cols.issubset(df.columns):
    print("\n[ERROR] labels.csv tidak memiliki kolom yang benar.")
    print("Kolom wajib:", required_cols)
    print("Kolom yang ada:", list(df.columns))
    raise SystemExit("Perbaiki struktur labels.csv dulu.")

print("\n[INFO] Contoh isi labels.csv:")
display(df.head())

print("\n[INFO] Jumlah data:", len(df))


# ---- 6. Helper: load & preprocess image ----
IMG_HEIGHT = 224
IMG_WIDTH = 224

def load_and_preprocess_image(img_path, img_height=IMG_HEIGHT, img_width=IMG_WIDTH):
    """
    Baca gambar dari path, resize ke ukuran tetap, normalisasi ke [0,1].
    Return: numpy array shape (img_height, img_width, 3)
    """
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Image not found or unreadable: {img_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_width, img_height))
    img = img.astype("float32") / 255.0
    return img


# ---- 7. Load semua gambar ke memory (untuk prototype) ----
X = []
y = []

missing_files = []

for idx, row in df.iterrows():
    fname = str(row["filename"])
    muac = row["muac_cm"]

    img_path = os.path.join(IMAGES_DIR, fname)
    if not os.path.exists(img_path):
        missing_files.append(fname)
        continue

    try:
        img = load_and_preprocess_image(img_path)
        X.append(img)
        y.append(float(muac))
    except Exception as e:
        print(f"[WARNING] Gagal memproses {fname}: {e}")

if missing_files:
    print("\n[WARNING] Ada filename di labels.csv yang tidak ditemukan di folder images/:")
    for f in missing_files[:10]:
        print(" -", f)
    if len(missing_files) > 10:
        print(f"  ... dan {len(missing_files)-10} file lainnya.")

X = np.array(X, dtype="float32")
y = np.array(y, dtype="float32")

print("\n[INFO] Jumlah sampel yang berhasil dibaca:", X.shape[0])

if X.shape[0] < 20:
    print("\n[WARNING] Jumlah data sangat sedikit (<20).")
    print("Secara teknis masih bisa lanjut training, tetapi hasil model kemungkinan sangat buruk.")
    print("Disarankan kumpulkan lebih banyak data dulu.")
    # Kalau mau stop otomatis, aktifkan baris di bawah:
    # raise SystemExit("Stop: data terlalu sedikit untuk training yang masuk akal.")


# ---- 8. Split Train / Validation / Test ----
test_size = 0.1
val_size = 0.2  # relatif terhadap data train+val

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=test_size, random_state=42
)

val_relative = val_size / (1.0 - test_size)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=val_relative, random_state=42
)

print("\n[INFO] Split data:")
print("  Train :", X_train.shape[0])
print("  Val   :", X_val.shape[0])
print("  Test  :", X_test.shape[0])


# ---- 9. Define MUAC regression model (MobileNetV2 backbone) ----
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
)

# Untuk awal, kita freeze base_model (transfer learning basic)
base_model.trainable = False

inputs = layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="linear", name="muac_cm")(x)

model = models.Model(inputs, outputs, name="NutriScan_MUAC_Regression")

model.summary()


# ---- 10. Compile model ----
# MAE (Mean Absolute Error) = rata-rata selisih cm antara prediksi & label
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mae",
    metrics=["mae"]
)

# ---- 11. Training ----
EPOCHS = 20
BATCH_SIZE = 16

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

# ---- 12. Evaluate on test set ----
test_loss, test_mae = model.evaluate(X_test, y_test, verbose=1)
print("\n[RESULT] Test MAE (cm):", test_mae)


# ---- 13. Save model ----
model_filename = "muac_model_v1.keras"  # format baru Keras
save_path = os.path.join(MODELS_DIR, model_filename)
model.save(save_path)
print(f"\n[INFO] Model saved to: {save_path}")
print("[DONE] Training pipeline selesai. Model siap dipakai untuk inference.")


CURRENT_DIR: c:\Users\Christian\Documents\AOLAI\NutriScanAI\notebooks
BASE_DIR   : c:\Users\Christian\Documents\AOLAI\NutriScanAI
DATASET_DIR: c:\Users\Christian\Documents\AOLAI\NutriScanAI\dataset
IMAGES_DIR : c:\Users\Christian\Documents\AOLAI\NutriScanAI\dataset\images
LABELS_PATH: c:\Users\Christian\Documents\AOLAI\NutriScanAI\dataset\labels.csv
MODELS_DIR : c:\Users\Christian\Documents\AOLAI\NutriScanAI\models

[INFO] Contoh isi labels.csv:


,filename,category,muac_cm
0,Malnourished_1.jpg,Malnourished,11.0
1,Malnourished_2.jpg,Malnourished,11.2
2,Malnourished_3.jpg,Malnourished,11.3
3,Malnourished_4.jpg,Malnourished,11.4
4,Malnourished_5.jpg,Malnourished,11.5



[INFO] Jumlah data: 14

[INFO] Jumlah sampel yang berhasil dibaca: 14

[WARNING] Jumlah data sangat sedikit (<20).
Secara teknis masih bisa lanjut training, tetapi hasil model kemungkinan sangat buruk.
Disarankan kumpulkan lebih banyak data dulu.

[INFO] Split data:
  Train : 9
  Val   : 3
  Test  : 2


Model: "NutriScan_MUAC_Regression"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ muac_cm (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - loss: 14.2167 - mae: 14.2167 - val_loss: 10.7252 - val_mae: 10.7252
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step - loss: 11.0052 - mae: 11.0052 - val_loss: 7.8456 - val_mae: 7.8456
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step - loss: 8.1675 - mae: 8.1675 - val_loss: 5.1901 - val_mae: 5.1901
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step - loss: 5.9491 - mae: 5.9491 - val_loss: 2.7607 - val_mae: 2.7607
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step - loss: 3.2920 - mae: 3.2920 - val_loss: 0.8123 - val_mae: 0.8123
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - loss: 0.9805 - mae: 0.9805 - val_loss: 1.4673 - val_mae: 1.4673
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step - loss: 1.9950 - mae: 1.9950 - val_loss: 2.6684 - val_mae: 2.6684
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step - loss: 2.2701 - mae: 2.2701 - val_loss: 3.0936 - val_mae: 3.0936
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step - loss: 2.9139 - ma